
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>



# LAB - Building Multi-stage AI System

In this lab, you will construct a multi-stage reasoning system using Databricks' features and LangChain.

You will start by building the first chain, which performs a search using a dataset containing product descriptions from Etsy. Following that, you will create the second chain, which creates an image for the proposed product. Finally, you will integrate these chains to form a complete multi-stage AI system.


**Lab Outline:**

In this lab, you will need to complete the following tasks;

* **Task 1:** Create a Vector Store

* **Task 2:** Build the First Chain (Vector Store Search)

* **Task 3:** Build the Second Chain (Product Image)

* **Task 4:**  Integrate Chains into a Multi-chain System

**📝 Your task:** Complete the **`<FILL_IN>`** sections in the code blocks and follow the other steps as instructed.

## REQUIRED - SELECT CLASSIC COMPUTE
Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:
1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

2. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

   - Click **More** in the drop-down.
   
   - In the **Attach to an existing compute resource** window, use the first drop-down to select your unique cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

2. Find the triangle icon to the right of your compute cluster name and click it.

3. Wait a few minutes for the cluster to start.

4. Once the cluster is running, complete the steps above to select your cluster.

## Requirements

Please review the following requirements before starting the lesson:

* To run this notebook, you need to use one of the following Databricks runtime(s): **17.3.x-cpu-ml-scala2.13**


## Classroom Setup

Before starting the lab, run the provided classroom setup script. This script will define configuration variables necessary for the lab. Execute the following cell:

In [0]:
%pip install -U -qqq databricks-sdk databricks-vectorsearch==0.60 'mlflow-skinny[databricks]==3.4.0' databricks-langchain==0.8.0 langchain==0.3.7 langchain-community==0.3.7 youtube_search==2.1.2 Wikipedia==1.4.0 
%restart_python

/databricks/python_shell/lib/lsp_backend/line_magic_sanitizer.py:98: UserWarning: `make_tokens_by_line` received a list of lines which do not have lineending markers ('\n', '\r', '\r\n', '\x0b', '\x0c'), behavior will be unspecified
  tokens = make_tokens_by_line(lines)
/databricks/python_shell/lib/lsp_backend/line_magic_sanitizer.py:98: UserWarning: `make_tokens_by_line` received a list of lines which do not have lineending markers ('\n', '\r', '\r\n', '\x0b', '\x0c'), behavior will be unspecified
  tokens = make_tokens_by_line(lines)


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%run ../Includes/Classroom-Setup-02LAB


The examples and models presented in this course are intended solely for demonstration and educational purposes.
 Please note that the models and prompt examples may sometimes contain offensive, inaccurate, biased, or harmful content.


/databricks/python_shell/lib/lsp_backend/line_magic_sanitizer.py:98: UserWarning: `make_tokens_by_line` received a list of lines which do not have lineending markers ('\n', '\r', '\r\n', '\x0b', '\x0c'), behavior will be unspecified
  tokens = make_tokens_by_line(lines)


**Other Conventions:**

Throughout this demo, we'll refer to the object `DA`. This object, provided by Databricks Academy, contains variables such as your username, catalog name, schema name, working directory, and dataset locations. Run the code block below to view these details:

In [0]:
print(f"Username:          {DA.username}")
print(f"Catalog Name:      {DA.catalog_name}")
print(f"Schema Name:       {DA.schema_name}")
print(f"Working Directory: {DA.paths.working_dir}")
print(f"Dataset Location:  {DA.paths.datasets}")

Username:          labuser12420723_1761941234@vocareum.com
Catalog Name:      dbacademy
Schema Name:       labuser12420723_1761941234
Working Directory: /Volumes/dbacademy/ops/labuser12420723_1761941234@vocareum_com
Dataset Location:  NestedNamespace (dais='/Volumes/dbacademy_dais/v01', docs='/Volumes/dbacademy_docs/v01')


## Load Dataset

Before you start building the AI chain, you need to load and prepare the dataset and save it as a Delta table.  
For this demo, we will use the **[Databricks Documentation Dataset](/marketplace/consumer/listings/03bbb5c0-983d-4523-833a-57e994d76b3b?o=1120757972560637)** available from the Databricks Marketplace.

This dataset contains documentation pages with associated `id`, `url`, and `content`.  
We will format the data to create a single unified `document` field combining the URL and content, which will then be used to build a Vector Store.

The table will be created for you in the next code block.

In [0]:
## Load the docs table from Unity Catalog
vs_source_table_fullname = f"{DA.catalog_name}.{DA.schema_name}.docs"
create_docs_table(vs_source_table_fullname)
## Display a sample of the data
display(spark.sql(f"SELECT * FROM {vs_source_table_fullname}"))

Validation of table dbacademy_docs.v01.docs complete. No errors found.


id document 25277 ## URL: https://docs.databricks.com/en/ingestion/bad-records.html

## Content: Handle bad records and files 
Databricks provides a number of options for dealing with files that contain bad records. Examples of bad data include: 
Incomplete or corrupt records: Mainly observed in text based file formats like JSON and CSV. For example, a JSON record that doesn’t have a closing brace or a CSV record that doesn’t have as many columns as the header or first record of the CSV file. 
Mismatched data types: When the value for a column doesn’t have the specified or inferred data type. 
Bad field names: Can happen in all file formats, when the column name specified in the file or record has a different casing than the specified or inferred schema. 
Corrupted files: When a file cannot be read, which might be due to metadata or data corruption in binary file types such as Avro, Parquet, and ORC. On rare occasion, might be caused by long-lasting transient failures in the underlying storage system. 
Missing files: A file that was discovered during query analysis time and no longer exists at processing time. 
Use badRecordsPath
Use badRecordsPath
When you set badRecordsPath, the specified path records exceptions for bad records or files encountered during data loading. 
In addition to corrupt records and files, errors indicating deleted files, network connection exception, IO exception, and so on are ignored and recorded under the badRecordsPath. 
Note 
Using the badRecordsPath option in a file-based data source has a few important limitations: 
It is non-transactional and can lead to inconsistent results. 
Transient errors are treated as failures.

Unable to find input file
Unable to find input file
val df = spark.read .option("badRecordsPath", "/tmp/badRecordsPath") .format("parquet").load("/input/parquetFile") // Delete the input parquet file '/input/parquetFile' dbutils.fs.rm("/input/parquetFile") df.show() 
In the above example, since df.show() is unable to find the input file, Spark creates an exception file in JSON format to record the error. For example, /tmp/badRecordsPath/20170724T101153/bad_files/xyz is the path of the exception file. This file is under the specified badRecordsPath directory, /tmp/badRecordsPath. 20170724T101153 is the creation time of this DataFrameReader. bad_files is the exception type. xyz is a file that contains a JSON record, which has the path of the bad file and the exception/reason message.

Input file contains bad record
Input file contains bad record
// Creates a json file containing both parsable and corrupted records Seq("""{"a": 1, "b": 2}""", """{bad-record""").toDF().write.format("text").save("/tmp/input/jsonFile") val df = spark.read .option("badRecordsPath", "/tmp/badRecordsPath") .schema("a int, b int") .format("json") .load("/tmp/input/jsonFile") df.show() 
In this example, the DataFrame contains only the first parsable record ({"a": 1, "b": 2}). The second bad record ({bad-record) is recorded in the exception file, which is a JSON file located in /tmp/badRecordsPath/20170724T114715/bad_records/xyz. The exception file contains the bad record, the path of the file containing the record, and the exception/reason message. After you locate the exception files, you can use a JSON reader to process them. 25278 ## URL: https://docs.databricks.com/en/ingestion/copy-into/configure-data-access.html

## Content: Configure data access for ingestion 
This article describes how admin users can configure access to data in a bucket in Amazon S3 (S3) so that Databricks users can load data from S3 into a table in Databricks. 
This article describes the following ways to configure secure access to source data: 
(Recommended) Create a Unity Catalog volume. 
Create a Unity Catalog external location with a storage credential. 
Launch a compute resource that uses an AWS instance profile. 
Generate temporary credentials (an AWS access key ID, a secret key, and a session token). 
Before you begin
Befor

%md 
## Create a Vector Store

In this step, you will compute embeddings for the dataset containing information about the products and store them in a Vector Search index using Databricks Vector Search.

**🚨IMPORTANT: Vector Search endpoints must be created before running the rest of the demo. These are already created for you in Databricks Lab environment.**


In [0]:
## Assign Vector Search endpoint by username
vs_endpoint_prefix = "vs_endpoint_"
vs_endpoint_name = vs_endpoint_prefix + str(get_fixed_integer(DA.unique_name("_")))
print(f"Assigned Vector Search endpoint name: {vs_endpoint_name}.")

Assigned Vector Search endpoint name: vs_endpoint_4.


In [0]:
## Index table name
vs_index_table_fullname = f"{DA.catalog_name}.{DA.schema_name}.doc_embeddings"

## Store embeddings in vector store
## NOTE: we're using 'content' as the embedding column
create_vs_index(vs_endpoint_name, vs_index_table_fullname, vs_source_table_fullname, "document" )

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Endpoint named vs_endpoint_4 is ready.


## Task 1: Build the First Chain (Vector Store Search)

In this task, you will create first chain that will search for product details from the Vector Store using a dataset containing product descriptions.

**Instructions:**
   - Configure components for the first chain to perform a search using the Vector Store.
   - Utilize the loaded dataset to generate prompts for Vector Store search queries.
   - Set up retrieval to extract relevant product details based on the generated prompts and search results.


In [0]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.prompts import PromptTemplate
from databricks_langchain import ChatDatabricks, DatabricksVectorSearch
from langchain_core.output_parsers import StrOutputParser

## Define the Databricks Chat model: llama-3
llm_llama = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", max_tokens = 1000)

## Define the prompt template for generating search queries
prompt_template_vs = PromptTemplate.from_template(
    """
    You are a documentation assistant. Based on the following context from a technical document, generate a concise summary or relevant content snippet for answering the user’s question.

    Write a response that is aligned with the tone and format of technical documentation and helps the user understand or resolve their query.

    Maximum 300 words.

    Use the following document snippet and context as example;

    <context>
    {context}
    </context>

    Question: {input}
    """
)

## Construct the RetrievalQA chain for Vector Store search
def get_retriever(persist_dir=None):
    vsc = VectorSearchClient(disable_notice=True)
    vs_index = vsc.get_index(vs_endpoint_name, vs_index_table_fullname)
    vectorstore = DatabricksVectorSearch(vs_index_table_fullname)
    return vectorstore.as_retriever(search_kwargs={'k': 3})

## Construct the chain for question-answering
question_answer_chain = create_stuff_documents_chain(llm_llama, prompt_template_vs)
chain1 = create_retrieval_chain(get_retriever(), question_answer_chain)

## Invoke the chain with an example query   
response = chain1.invoke({"input": "How do I register the ML model from workspace A to Unity Catalog in workspace B ?"})
print(response['answer'])

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
**Registering an ML Model from Workspace A to Unity Catalog in Workspace B**

To register an ML model from Workspace A to Unity Catalog in Workspace B, follow these steps:

### Step 1: Set the Registry URI

In Workspace A, set the registry URI to the Unity Catalog in Workspace B using the `mlflow.set_registry_uri()` method:
```python
import mlflow
mlflow.set_registry_uri("databricks-uc")
```
### Step 2: Register the Model

Register the ML model in Workspace A using the `mlflow.log_model()` method:
```python
mlflow.log_model(model, "model_name")
```
### Step 3: Copy the Model Version

Copy the model version from Workspace A to Unity Catalog in Workspace B using the `client.copy_model_version()` method:
```python
from mlflow.client import MlflowClient
client = MlflowClient()
client

Trace(trace_id=tr-160d3f6eefe69af8b17c83920dd9e8df)

## Task 2: Build the Second Chain (Optimization)

In this step, you will create a second chain to enhance the product details generated by the first chain. This optimization process aims to make the descriptions more compelling and SEO-friendly. In a real-world scenario, this model could be trained on your internal data or fine-tuned to align with your specific business objectives.

**Instructions:**

- Define a second chain using `llama-3-70b-instruct`.  

- Create a prompt to optimize the generated product description. For example:  
  *"You are a marketing expert. Revise the product title and description to be SEO-friendly and more appealing to Databricks users."*

- Use `product_details` as the parameter to be passed into the prompt.  

- Implement the chain and test it with a sample input.  


In [0]:
## Define the Databricks Chat model using llama-3-3-70b-instruct
llm_llama3 = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", max_tokens = 1000)

## Define the prompt template for refining documentation output
# doc_optimization_prompt = PromptTemplate.from_template(
#     """
#     You are a technical writer. Improve the following documentation snippet to make it clearer, concise, and aligned with the tone used in Databricks documentation.

#     Documentation snippet: {product_details}

#     Return only the revised documentation content.
#     """
# )

doc_optimization_prompt = PromptTemplate.from_template(
    """
    You are a technical writer. Revise the following documentation snippet to be clear, concise, and aligned with the tone and style of Databricks documentation.

    Documentation snippet: {product_details}

    Return only the improved version of the documentation.
    """
)

## Define chain 2
chain2 = doc_optimization_prompt | llm_llama3 | StrOutputParser()

## Test the chain
chain2.invoke({"product_details": "Query testing product with mobile app control"})

/databricks/python_shell/lib/lsp_backend/line_magic_sanitizer.py:98: UserWarning: `make_tokens_by_line` received a list of lines which do not have lineending markers ('\n', '\r', '\r\n', '\x0b', '\x0c'), behavior will be unspecified
  tokens = make_tokens_by_line(lines)


'# Query Testing with Mobile App Control\n\n## Overview\n\nThis feature enables you to test and validate queries using a mobile app, providing a seamless and efficient way to ensure data accuracy and integrity.\n\n## Prerequisites\n\n* A Databricks account with the necessary permissions\n* A mobile device with the Databricks mobile app installed\n* A query to be tested\n\n## Testing Queries with Mobile App Control\n\n1. **Launch the Mobile App**: Open the Databricks mobile app on your device and log in to your account.\n2. **Select the Query**: Choose the query you want to test from the list of available queries.\n3. **Run the Query**: Tap the "Run" button to execute the query and retrieve the results.\n4. **Verify the Results**: Review the query results on your mobile device to ensure they are accurate and match your expectations.\n5. **Refine and Rerun**: If necessary, refine the query and rerun it to validate the changes.\n\n## Best Practices\n\n* Ensure your mobile device has a sta

Trace(trace_id=tr-45d438a5ae621caab2b46a14d1113cbc)

## Task 3: Integrate Chains into a Multi-chain System

In this task, you will link the individual chains created in Task 2 and Task 3 together to form a multi-chain system that can handle multi-stage reasoning.

**Instructions:**

- Use Databricks **`Llama Chat model`** for processing text inputs, which is defined above in the first task.

- Create a prompt template to generate an **`HTML page`** for displaying generated product details.

- Construct the **`Multi-Chain System`**  by combining the outputs of the previous chains. **Important**: You will need to rename the output of the first chain and second chain while passing them to the next stage. This sequential chain should be as; **chain3 = chain1 > (`product_details`) > chain2 > `(optimized_product_details)` > prompt3**.  

- Invoke the multi-chain system with the input data to generate the HTML page for the specified product.


In [0]:
from langchain.schema.runnable import RunnablePassthrough, RunnableMap
from langchain_core.output_parsers import StrOutputParser
from IPython.display import display, HTML

## Define the prompt template for generating the HTML page
prompt_template_3 = PromptTemplate.from_template(
  """Create an HTML section for the following technical documentation snippet:
    
  Content: {optimized_doc}

  Return valid HTML (no head/body tags).
  """
)


# Construct the multi-chain system
chain3 = (
  chain1 | 
  RunnableMap({"product_details": lambda x: x["answer"]}) | 
  chain2 | 
  RunnableMap({"optimized_doc": lambda x: x}) |
  prompt_template_3 |  
  llm_llama | 
  StrOutputParser()
)

## Sample query
query = {"input": "How do I create a Delta table in Databricks?"}
# query = {"input": "How do I register the ML model from workspace A to Unity Catalog in workspace B "}

output_html = chain3.invoke(query)

## Display the generated HTML output
display(HTML(output_html))

/databricks/python_shell/lib/lsp_backend/line_magic_sanitizer.py:98: UserWarning: `make_tokens_by_line` received a list of lines which do not have lineending markers ('\n', '\r', '\r\n', '\x0b', '\x0c'), behavior will be unspecified
  tokens = make_tokens_by_line(lines)


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


Trace(trace_id=tr-0b1eccea13dd551daaa81e287f99ef72)

In [0]:
query = {"input": "How do I register the ML model from workspace A to Unity Catalog in workspace B "}

output_html = chain3.invoke(query)

## Display the generated HTML output
display(HTML(output_html))

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


Trace(trace_id=tr-032f563dba913e1b3c1ada9e4e34f7da)

## Task 4: Save the Chain to Model Registry in UC

In this task, you will save the multi-stage chain system within our Unity Catalog.

**Instructions:**

- Set the model registry to UC and use the model name defined.

- Log and register the final multi-chain system.

- To test the registered model, load the model back from model registry and query it using a sample query. 

After registering the chain, you can view the chain and models in the **Catalog Explorer**.

In [0]:
from mlflow.models import infer_signature
import mlflow


## Set model registry to UC
mlflow.set_registry_uri("databricks-uc")
model_name = f"{DA.catalog_name}.{DA.schema_name}.multi_stage_doc_chain"

## Log the model
with mlflow.start_run(run_name="multi_stage_doc_chain") as run:
    signature = infer_signature(query, output_html)
    model_info = mlflow.langchain.log_model(
        chain3,
        loader_fn=get_retriever,
        name="chain",
        registered_model_name=model_name,
        input_example=query,
        signature=signature
    )

## Load and test the model
model_uri = f"models:/{model_name}/{model_info.registered_model_version}"
model = mlflow.langchain.load_model(model_uri)

output_html = model.invoke(query)
display(HTML(output_html))

🔗 View Logged Model at: https://dbc-ec8d736c-20ed.cloud.databricks.com/ml/experiments/3632639334096082/models/m-5f61f27b1c27403cac218a4329cfda8f?o=3843975142727865
2025/10/31 22:56:16 INFO mlflow: Attempting to auto-detect Databricks resource dependencies for the current langchain model. Dependency auto-detection is best-effort and may not capture all dependencies of your langchain model, resulting in authorization errors when serving or querying your model. We recommend that you explicitly pass `resources` to mlflow.langchain.log_model() to ensure authorization to dependent resources succeeds when the model is deployed.
2025/10/31 22:56:46 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-5f61f27b1c27403cac218a4329cfda8f
2025/10/31 22:56:46 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


Successfully registered model 'dbacademy.labuser12420723_1761941234.multi_stage_doc_chain'.


Uploading artifacts:   0%|          | 0/46 [00:00<?, ?it/s]

🔗 Created version '1' of model 'dbacademy.labuser12420723_1761941234.multi_stage_doc_chain': https://dbc-ec8d736c-20ed.cloud.databricks.com/explore/data/models/dbacademy/labuser12420723_1761941234/multi_stage_doc_chain/version/1?o=3843975142727865


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


Trace(trace_id=tr-c778df7e33545285adc47b0fa8b8838c)


## Conclusion

In this lab, you've learned how to build a multi-stage AI system using Databricks and LangChain. By integrating multiple chains, you can perform complex reasoning tasks such as searching for product details and optimizing the response based on your business needs. This approach enables the development of sophisticated AI systems capable of handling diverse tasks efficiently.


&copy; 2025 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>